# Week 9: Transfer Learning

**October 13:** Fall Break — no class  
**Lecture 15 — October 15:** Transfer Learning

Transfer learning starts from representations learned on a large source dataset and adapts them to a target task. We compare the same ResNet-18 architecture under three initialization and training strategies: training from scratch, frozen feature extraction, and fine-tuning.

## Learning goals

By the end of the lecture, you should be able to:

- explain why representations learned on ImageNet can transfer;
- replace a pretrained classifier for a new label space;
- distinguish feature extraction from fine-tuning;
- control trainable parameters with `requires_grad`;
- handle normalization and BatchNorm correctly while freezing layers;
- use smaller, discriminative learning rates when fine-tuning; and
- recognize domain shift and negative transfer.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable; full training will be slow.")

## Data and pretrained preprocessing

The pretrained weights were learned from ImageNet images normalized with the ImageNet channel means and standard deviations. Matching that preprocessing is part of using the weights correctly. The training pipeline adds random crops, flips, and RandAugment; validation is deterministic.

The dataset remains outside the repository at `../../datasets/imagenette2`. Imagenette is especially convenient here: its ten classes are drawn from ImageNet, so it represents a favorable, low-domain-shift transfer setting.

In [ ]:
DATA_ROOT = Path("../../datasets/imagenette2")
CHECKPOINT_DIRECTORY = Path("checkpoints")
CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 224
BATCH_SIZE = 64
SCRATCH_EPOCHS = 40
FEATURE_EXTRACTION_EPOCHS = 15
FINE_TUNING_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 6
MINIMUM_IMPROVEMENT = 1e-4

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)
training_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
evaluation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
training_dataset = datasets.ImageFolder(DATA_ROOT / "train", training_transform)
validation_dataset = datasets.ImageFolder(DATA_ROOT / "val", evaluation_transform)
loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": 2,
    "pin_memory": device.type == "cuda",
    "persistent_workers": True,
}
training_loader = DataLoader(training_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
number_of_classes = len(training_dataset.classes)
print(f"training examples:   {len(training_dataset):,}")
print(f"validation examples: {len(validation_dataset):,}")
print("classes:", training_dataset.classes)

## Three strategies

| Strategy | Initialization | Updated parameters | Typical use |
|---|---|---|---|
| From scratch | Random | Entire model | Large target dataset or very different domain |
| Feature extraction | Pretrained | New classifier only | Small target dataset, fast baseline |
| Fine-tuning | Pretrained | Some or all pretrained layers | More target data and a related domain |

Feature extraction treats the backbone as a fixed mapping $h=f_\theta(X)$ and learns only a target classifier $g_\phi(h)$. Fine-tuning also updates some of $\theta$, usually with a smaller step size so useful source features are not immediately destroyed.

## Constructing and freezing the model

Torchvision's weights object identifies both the learned parameters and their metadata. Replacing `model.fc` creates a new randomly initialized ten-class head. Freezing is controlled by `requires_grad`; the optimizer should receive only parameters that remain trainable.

Freezing weights does **not** automatically freeze BatchNorm running means and variances. During feature extraction, frozen BatchNorm modules are put in evaluation mode after `model.train()` so their pretrained statistics remain fixed.

In [ ]:
def replace_resnet_classifier(model, number_of_classes):
    model.fc = nn.Linear(model.fc.in_features, number_of_classes)
    return model


def set_trainable(module, trainable):
    for parameter in module.parameters():
        parameter.requires_grad = trainable


def set_frozen_batch_norm_to_eval(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            local_parameters = list(module.parameters(recurse=False))
            if local_parameters and not any(
                parameter.requires_grad for parameter in local_parameters
            ):
                module.eval()


def count_parameters(model):
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    )
    return trainable, total


def print_parameter_counts(label, model):
    trainable, total = count_parameters(model)
    print(f"{label:<24} {trainable:>12,} / {total:>12,} trainable / total")

## Shared training loop

All three experiments use the same data split, augmentation, loss, validation metric, early-stopping rule, and best-checkpoint behavior. This does not make the comparison scientifically exhaustive—the learning rates and epoch budgets appropriately differ by strategy—but it keeps the major evaluation choices controlled.

In [ ]:
def run_epoch(model, loader, loss_function, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if training:
        set_frozen_batch_norm_to_eval(model)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(X)
            loss = loss_function(logits, y)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
        total_loss += loss.item() * X.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_examples += X.size(0)
    return total_loss / total_examples, total_correct / total_examples


def fit_model(
    model, optimizer, scheduler, epochs, checkpoint_path,
    patience=EARLY_STOPPING_PATIENCE,
    minimum_improvement=MINIMUM_IMPROVEMENT,
):
    loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_state = deepcopy(model.state_dict())
    best_validation_loss = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    history = []
    print(
        f"{'epoch':>7} {'loss':>11} {'val_loss':>11} "
        f"{'accuracy':>11} {'val_accuracy':>14} {'lr':>10}  status"
    )
    print("-" * 84)
    for epoch in range(1, epochs + 1):
        training_loss, training_accuracy = run_epoch(
            model, training_loader, loss_function, optimizer
        )
        validation_loss, validation_accuracy = run_epoch(
            model, validation_loader, loss_function
        )
        learning_rate = max(group["lr"] for group in optimizer.param_groups)
        improved = validation_loss < best_validation_loss - minimum_improvement
        status = (
            "best" if improved
            else f"wait {epochs_without_improvement + 1}/{patience}"
        )
        history.append({
            "epoch": epoch, "loss": training_loss,
            "val_loss": validation_loss, "accuracy": training_accuracy,
            "val_accuracy": validation_accuracy, "learning_rate": learning_rate,
        })
        print(
            f"{epoch:7d} {training_loss:11.5f} {validation_loss:11.5f} "
            f"{training_accuracy:11.4f} {validation_accuracy:14.4f} "
            f"{learning_rate:10.2e}  {status}"
        )
        scheduler.step()
        if improved:
            best_validation_loss = validation_loss
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
            torch.save({
                "epoch": epoch, "model_state_dict": best_state,
                "validation_loss": validation_loss,
                "validation_accuracy": validation_accuracy,
                "history": history,
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break
    model.load_state_dict(best_state)
    print(f"Restored epoch {best_epoch}; checkpoint: {checkpoint_path}")
    return history

## Experiment 1: train from scratch

This is the control condition. The ResNet-18 architecture is identical, but every parameter begins randomly initialized and must be learned from 9,469 Imagenette training images. It receives the largest epoch budget.

In [ ]:
scratch_model = replace_resnet_classifier(
    models.resnet18(weights=None), number_of_classes
).to(device)
print_parameter_counts("from scratch", scratch_model)
scratch_optimizer = torch.optim.AdamW(
    scratch_model.parameters(), lr=3e-4, weight_decay=1e-4
)
scratch_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    scratch_optimizer, T_max=SCRATCH_EPOCHS
)
scratch_history = fit_model(
    scratch_model, scratch_optimizer, scratch_scheduler, SCRATCH_EPOCHS,
    CHECKPOINT_DIRECTORY / "week9_resnet18_scratch_best.pt",
)

## Experiment 2: frozen feature extraction

The ImageNet backbone is frozen and only the new classifier is optimized. This is cheap: activations still pass through the backbone, but autograd does not store a backward graph for its parameters. A larger learning rate is reasonable for the newly initialized head.

The first use of pretrained weights may download them into PyTorch's local cache.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
feature_model = models.resnet18(weights=weights)
set_trainable(feature_model, False)
feature_model = replace_resnet_classifier(feature_model, number_of_classes).to(device)
print_parameter_counts("feature extraction", feature_model)
feature_optimizer = torch.optim.AdamW(
    feature_model.fc.parameters(), lr=1e-3, weight_decay=1e-4
)
feature_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    feature_optimizer, T_max=FEATURE_EXTRACTION_EPOCHS
)
feature_history = fit_model(
    feature_model, feature_optimizer, feature_scheduler,
    FEATURE_EXTRACTION_EPOCHS,
    CHECKPOINT_DIRECTORY / "week9_resnet18_feature_best.pt",
)

## Experiment 3: fine-tune the last residual stage

Fine-tuning continues from the best feature-extraction model. `layer4` is unfrozen while earlier stages remain fixed. The pretrained residual stage receives a smaller learning rate than the classifier head:

$$\alpha_{\text{layer4}} < \alpha_{\text{head}}.$$

These **discriminative learning rates** let the new head adapt quickly while changing useful pretrained features cautiously. Unfreezing more stages can help when there is enough target data, but increases compute and the risk of overfitting or catastrophic forgetting.

In [ ]:
fine_tune_model = feature_model
set_trainable(fine_tune_model, False)
set_trainable(fine_tune_model.layer4, True)
set_trainable(fine_tune_model.fc, True)
print_parameter_counts("fine-tuning layer4", fine_tune_model)
fine_tune_optimizer = torch.optim.AdamW([
    {"params": fine_tune_model.layer4.parameters(), "lr": 1e-4},
    {"params": fine_tune_model.fc.parameters(), "lr": 5e-4},
], weight_decay=1e-4)
fine_tune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    fine_tune_optimizer, T_max=FINE_TUNING_EPOCHS
)
fine_tune_history = fit_model(
    fine_tune_model, fine_tune_optimizer, fine_tune_scheduler,
    FINE_TUNING_EPOCHS,
    CHECKPOINT_DIRECTORY / "week9_resnet18_fine_tuned_best.pt",
)

## Compare the learning curves

Feature extraction should begin with substantially better validation performance and converge quickly. Fine-tuning may improve the pretrained baseline further. The scratch model often needs more updates before it learns useful low-level features. Compare best validation performance, convergence speed, trainable parameter count, and wall-clock cost—not only the last epoch.

In [ ]:
histories = {
    "scratch": scratch_history,
    "feature extraction": feature_history,
    "fine-tuning": fine_tune_history,
}
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, history in histories.items():
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["val_loss"] for row in history], label=label)
    axes[1].plot(epochs, [row["val_accuracy"] for row in history], label=label)
axes[0].set(xlabel="epoch", ylabel="validation loss", title="Validation loss")
axes[1].set(xlabel="epoch", ylabel="validation accuracy", title="Validation accuracy")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.25)
figure.tight_layout()

print(f"{'strategy':<22} {'best val loss':>14} {'best val acc':>14}")
print("-" * 52)
for label, history in histories.items():
    print(
        f"{label:<22} {min(row['val_loss'] for row in history):>14.5f} "
        f"{max(row['val_accuracy'] for row in history):>14.4f}"
    )

## Class-level evaluation

The fine-tuned model is evaluated by class below. Imagenette2 has no labeled test folder, and the validation split selected our checkpoints and hyperparameters. These results therefore describe validation behavior, not an unbiased final test estimate.

In [ ]:
@torch.no_grad()
def predict_labels(model, loader):
    model.eval()
    true_labels = []
    predicted_labels = []
    for X, y in loader:
        logits = model(X.to(device, non_blocking=True))
        true_labels.extend(y.numpy())
        predicted_labels.extend(logits.argmax(dim=1).cpu().numpy())
    return np.asarray(true_labels), np.asarray(predicted_labels)

true_labels, predicted_labels = predict_labels(
    fine_tune_model, validation_loader
)
print(classification_report(
    true_labels, predicted_labels,
    labels=range(number_of_classes),
    target_names=validation_dataset.classes,
    zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    true_labels, predicted_labels, labels=range(number_of_classes),
    display_labels=validation_dataset.classes,
    xticks_rotation=45, cmap="Blues",
)
plt.tight_layout()

## Domain shift and negative transfer

Transfer works best when source and target tasks share useful visual structure. Imagenette is extremely close to ImageNet, so strong transfer is expected. A target domain such as medical scans, satellite radar, or line drawings differs more in texture, color, scale, and semantics. Under substantial domain shift:

- the frozen representation may omit target-relevant information;
- unfreezing more layers may become necessary;
- source normalization may no longer be optimal;
- self-supervised pretraining on in-domain unlabeled data may help; and
- transfer can occasionally underperform a suitable model trained from scratch—**negative transfer**.

Always compare against a scratch baseline rather than assuming pretraining must help.

## Takeaways

- Pretrained weights and their preprocessing form one package.
- Feature extraction trains only a new head and is an efficient baseline.
- Fine-tuning adapts pretrained representations with smaller learning rates.
- Frozen BatchNorm statistics require explicit attention during training.
- Discriminative learning rates protect earlier features while adapting later layers.
- Domain similarity, dataset size, and compute determine the best transfer strategy.
- Validation-selected checkpoints must not be reported as unbiased test results.